# ModelCalculator: Test Notebook

Tests all major paths of `ModelCalculator.generate_predictions`:

| Section | What it exercises |
|---|---|
| 1 | Setup: synthetic data + adult income data |
| 2 | Basic predictions (no SHAP, no coefficients) |
| 3 | `subset_results=True` |
| 4 | `use_coefficients=True` (linear model) |
| 5 | `use_coefficients=True, include_contributions=True` |
| 6 | `global_coefficients=True` |
| 7 | `calculate_shap=True` |
| 8 | `calculate_shap=True, include_contributions=True` |
| 9 | `global_shap=True` |
| 10 | `global_shap=True, sample_size` (via direct call) |
| 11 | Conflict / ValueError guards |
| 12 | Adult income dataset: real model predictions |
| 13 | Adult income: global coefficients |
| 14 | Adult income: global SHAP |


## 1. Setup

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

import model_metrics

print(f"Model Metrics version: {model_metrics.__version__}")

from model_metrics.model_calculator import ModelCalculator

### 1a. Synthetic dataset

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="outcome")

X_train_df, X_test_df, y_train_s, y_test_s = train_test_split(
    X, y, test_size=0.2, random_state=42
)
y_test_df = y_test_s.to_frame()

In [ ]:
# Train models
model1 = LogisticRegression(random_state=42).fit(X_train_df, y_train_s)
model2 = RandomForestClassifier(random_state=42).fit(X_train_df, y_train_s)
model_titles = ["Logistic Regression", "Random Forest"]

# Build model_dict for a single outcome called "outcome"
# ModelCalculator expects {"model": {outcome_name: trained_model}}
outcomes = ["outcome"]

model_dict_lr = {"model": {"outcome": model1}}
model_dict_rf = {"model": {"outcome": model2}}

print("Models trained successfully.")

### 1b. Adult Income dataset

Update `model_path` and `data_path` to match your local directory structure.

In [ ]:
from eda_toolkit import ensure_directory

print(f"Model Metrics version: {model_metrics.__version__}")
print(f"Model Metrics authors: {model_metrics.__author__} \n")

## Define base paths
## `base_path`` represents the parent directory of current working directory
base_path = os.path.join(os.pardir)
## Go up one level from 'notebooks' to the parent directory, then into the
## 'results' folder

model_path = os.path.join(os.pardir, "model_files/results")
data_path = os.path.join(os.pardir, "model_files")
image_path_png = os.path.join(data_path, "images", "png_images")
image_path_svg = os.path.join(data_path, "images", "svg_images")

# Use the function to ensure the 'data' directory exists
ensure_directory(model_path)
ensure_directory(image_path_png)
ensure_directory(image_path_svg)

# ── helpers ──────────────────────────────────────────────────────────────────
import pickle


def loadObjects(path):
    with open(path, "rb") as f:
        return pickle.load(f)


# ── load pre-trained models ───────────────────────────────────────────────────
model_lr = loadObjects(os.path.join(model_path, "LogisticRegression.pkl"))
model_dt = loadObjects(os.path.join(model_path, "DecisionTreeClassifier.pkl"))
model_rf_ai = loadObjects(os.path.join(model_path, "RandomForestClassifier.pkl"))

# ── load test data ────────────────────────────────────────────────────────────
X_test_ai = pd.read_parquet(os.path.join(data_path, "X_test.parquet"))
y_test_ai = pd.read_parquet(os.path.join(data_path, "y_test.parquet"))

print(f"X_test_ai shape : {X_test_ai.shape}")
print(f"y_test_ai shape : {y_test_ai.shape}")
print(f"y_test_ai columns : {list(y_test_ai.columns)}")

In [ ]:
for label, m in [("LR", model_lr), ("DT", model_dt), ("RF", model_rf_ai)]:
    print(f"{label}: {type(m)}")
    print(f"  has predict:       {hasattr(m, 'predict')}")
    print(f"  has predict_proba: {hasattr(m, 'predict_proba')}")
    print(f"  has estimator:     {hasattr(m, 'estimator')}")
    print(f"  has steps:         {hasattr(m, 'steps')}")
    print(f"  has model:         {hasattr(m, 'model')}")
    print(f"  dir: {[a for a in dir(m) if not a.startswith('_')]}")
    print()

In [ ]:
for label, m in [("LR", model_lr), ("DT", model_dt), ("RF", model_rf_ai)]:
    print(f"{label} keys: {list(m.keys())}")
    for k, v in m.items():
        print(
            f"  [{k}]: {type(v)}  has predict={hasattr(v, 'predict')}  has predict_proba={hasattr(v, 'predict_proba')}"
        )
    print()

In [ ]:
# Adult income outcome name; adjust if your y_test column is named differently
AI_OUTCOME = y_test_ai.columns[0]

model_dict_ai_lr = {"model": {AI_OUTCOME: model_lr}}
model_dict_ai_rf = {"model": {AI_OUTCOME: model_rf_ai}}

outcomes_ai = [AI_OUTCOME]

print(f"Adult income outcome: '{AI_OUTCOME}'")

---
## 2. Basic predictions: no SHAP, no coefficients

In [ ]:
calc_lr = ModelCalculator(model_dict=model_dict_lr, outcomes=outcomes, top_n=3)

results_basic = calc_lr.generate_predictions(
    X_test=X_test_df,
    y_test=y_test_df,
)

print(results_basic.shape)
results_basic.head()

In [ ]:
# Sanity check: TP+FN+FP+TN should equal total rows
row_check = results_basic[["TP", "FN", "FP", "TN"]].sum(axis=1)
assert (row_check == 1).all(), "Each row should belong to exactly one quadrant"
print("TP/FN/FP/TN sanity check passed.")

# Confusion matrix summary
results_basic[["TP", "FN", "FP", "TN"]].sum()

---
## 3. `subset_results=True`

In [ ]:
results_subset = calc_lr.generate_predictions(
    X_test=X_test_df,
    y_test=y_test_df,
    subset_results=True,
)

print("Columns returned:", list(results_subset.columns))
assert set(results_subset.columns) == {"TP", "FN", "FP", "TN", "y_pred_proba"}
print("Subset columns assertion passed.")
results_subset.head()

---
## 4. `use_coefficients=True`: linear model, feature name lists

In [ ]:
results_coef = calc_lr.generate_predictions(
    X_test=X_test_df,
    y_test=y_test_df,
    use_coefficients=True,
)

coef_col = f"top_{calc_lr.top_n}_coefficients"
print(f"Coefficient column present: '{coef_col}' in {list(results_coef.columns)}")

# Each row should be a list of top-N feature names
sample = results_coef[coef_col].iloc[0]
print(f"Sample row value: {sample}")
assert isinstance(sample, list), "Expected a list of feature names"
assert (
    len(sample) == calc_lr.top_n
), f"Expected {calc_lr.top_n} features, got {len(sample)}"
print("Coefficient list assertion passed.")

---
## 5. `use_coefficients=True, include_contributions=True`: contribution dicts

In [ ]:
results_coef_contrib = calc_lr.generate_predictions(
    X_test=X_test_df,
    y_test=y_test_df,
    use_coefficients=True,
    include_contributions=True,
)

sample_contrib = results_coef_contrib[coef_col].iloc[0]
print(f"Sample row value: {sample_contrib}")
assert isinstance(sample_contrib, dict), "Expected a dict of {feature: contribution}"
assert len(sample_contrib) == calc_lr.top_n
print("Contribution dict assertion passed.")

---
## 6. `global_coefficients=True`

In [ ]:
global_coef_df = calc_lr.generate_predictions(
    X_test=X_test_df,
    y_test=y_test_df,
    global_coefficients=True,
)

print(type(global_coef_df))
assert isinstance(global_coef_df, pd.DataFrame)
assert "Feature" in global_coef_df.columns
assert "Coefficient" in global_coef_df.columns
print("Global coefficients assertion passed.")
global_coef_df

---
## 7. `calculate_shap=True`: top-N feature name lists

> Uses LogisticRegression (linear model, fast SHAP).

In [ ]:
results_shap = calc_lr.generate_predictions(
    X_test=X_test_df,
    y_test=y_test_df,
    calculate_shap=True,
)

shap_col = f"top_{calc_lr.top_n}_features"
print(f"SHAP column present: '{shap_col}'")

sample_shap = results_shap[shap_col].iloc[0]
print(f"Sample row value: {sample_shap}")
assert isinstance(sample_shap, list)
assert len(sample_shap) == calc_lr.top_n
print("SHAP feature list assertion passed.")

---
## 8. `calculate_shap=True, include_contributions=True`: contribution dicts

In [ ]:
results_shap_contrib = calc_lr.generate_predictions(
    X_test=X_test_df,
    y_test=y_test_df,
    calculate_shap=True,
    include_contributions=True,
)

sample_shap_contrib = results_shap_contrib[shap_col].iloc[0]
print(f"Sample row value: {sample_shap_contrib}")
assert isinstance(sample_shap_contrib, dict)
assert len(sample_shap_contrib) == calc_lr.top_n
print("SHAP contribution dict assertion passed.")

---
## 9. `global_shap=True`: batched global SHAP

In [ ]:
global_shap_df = calc_lr.generate_predictions(
    X_test=X_test_df,
    y_test=y_test_df,
    global_shap=True,
)

print(type(global_shap_df))
assert isinstance(global_shap_df, pd.DataFrame)
assert "Feature" in global_shap_df.columns
assert "SHAP Value" in global_shap_df.columns
assert len(global_shap_df) == X_test_df.shape[1], "Should have one row per feature"
print("Global SHAP assertion passed.")
global_shap_df

---
## 10. `global_shap=True` with `sample_size` via direct `_calculate_shap_values` call

In [ ]:
# Call the internal method directly to exercise the sample_size path
global_shap_sampled = calc_lr._calculate_shap_values(
    model=model1,
    X_test_m=X_test_df,
    global_shap=True,
    sample_size=50,  # only use 50 rows
)

assert isinstance(global_shap_sampled, pd.DataFrame)
assert "Feature" in global_shap_sampled.columns
print("global_shap with sample_size assertion passed.")
global_shap_sampled

---
## 11. Conflict / ValueError guards

In [ ]:
import traceback

conflict_cases = [
    dict(calculate_shap=True, use_coefficients=True),
    dict(global_shap=True, global_coefficients=True),
    dict(global_coefficients=True, subset_results=True),
    dict(global_shap=True, subset_results=True),
]

for kwargs in conflict_cases:
    try:
        calc_lr.generate_predictions(
            X_test=X_test_df,
            y_test=y_test_df,
            **kwargs,
        )
        print(f"FAIL: no error raised for {kwargs}")
    except ValueError as e:
        print(f"PASS: ValueError raised for {list(kwargs.keys())}: {e}")

---
## 12. Adult income dataset: basic predictions (all three models)

In [ ]:
for label, md in [
    ("Logistic Regression", {"model": {AI_OUTCOME: model_lr}}),
    ("Decision Tree", {"model": {AI_OUTCOME: model_dt}}),
    ("Random Forest", {"model": {AI_OUTCOME: model_rf_ai}}),
]:
    calc = ModelCalculator(model_dict=md, outcomes=outcomes_ai, top_n=3)
    res = calc.generate_predictions(X_test=X_test_ai, y_test=y_test_ai)
    tp = res["TP"].sum()
    fn = res["FN"].sum()
    fp = res["FP"].sum()
    tn = res["TN"].sum()
    print(
        f"{label:25s}  TP={tp:4d}  FN={fn:4d}  FP={fp:4d}  TN={tn:4d}  shape={res.shape}"
    )

---
## 13. Adult income: global coefficients (logistic regression)

In [ ]:
calc_ai_lr = ModelCalculator(
    model_dict={"model": {AI_OUTCOME: model_lr}},
    outcomes=outcomes_ai,
    top_n=5,
)

global_coef_ai = calc_ai_lr.generate_predictions(
    X_test=X_test_ai,
    y_test=y_test_ai,
    global_coefficients=True,
)

assert "Coefficient" in global_coef_ai.columns
print(f"Top 10 features by absolute coefficient:")
global_coef_ai.head(10)

---
## 14. Adult income: global SHAP (logistic regression)

> This can take a few minutes on the full test set. Set `sample_size` via the direct call if you want a faster run.

In [ ]:
global_shap_ai = calc_ai_lr.generate_predictions(
    X_test=X_test_ai,
    y_test=y_test_ai,
    global_shap=True,
)

assert "SHAP Value" in global_shap_ai.columns
print(f"Top 10 features by absolute SHAP value:")
global_shap_ai.head(10)

In [ ]:
# Optional: sampled version for speed
global_shap_ai_sampled = calc_ai_lr._calculate_shap_values(
    model=model_lr,
    X_test_m=X_test_ai.select_dtypes(include=["number"]),
    global_shap=True,
    sample_size=200,
)

global_shap_ai_sampled.head(10)

---
## 15. Adult income: row-wise SHAP with `subset_results`

> Returns only TP/FN/FP/TN/y_pred_proba + top-N SHAP feature column.

In [ ]:
results_ai_shap_subset = calc_ai_lr.generate_predictions(
    X_test=X_test_ai,
    y_test=y_test_ai,
    calculate_shap=True,
    subset_results=True,
)

print("Columns:", list(results_ai_shap_subset.columns))
results_ai_shap_subset.head()